# 03 - Walker Constellation

Third skill: constellation construction from a seed orbit.

You will learn how to:
- build Walker delta and star patterns
- choose two-body vs numerical member construction
- inspect satellite state outputs


In [ ]:
# Ensure local package import when running from this examples/ folder.
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_repo_root = _cwd if (_cwd / "nstk").is_dir() else _cwd.parent
if (_repo_root / "nstk").is_dir() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [ ]:
import numpy as np
from astropy.time import Time

from nstk.propagation import Orbit, build_walker_constellation

np.set_printoptions(precision=6, suppress=True)


In [ ]:
# 1) Build a seed orbit
seed = Orbit.from_kepler_two_body(
    epoch=Time("2026-01-01T00:00:00", scale="utc"),
    a=7000e3,
    e=0.001,
    i=np.deg2rad(53.0),
    raan=np.deg2rad(20.0),
    argp=np.deg2rad(15.0),
    anomaly=np.deg2rad(10.0),
    anomaly_type="mean",
)

seed.propagator.getClass().getSimpleName()


In [ ]:
# 2) Walker delta: T/P/F = 24/6/1
delta = build_walker_constellation(
    seed,
    total_satellites=24,
    num_planes=6,
    phasing=1,
    pattern="delta",
    include_seed=True,
)

print("delta count:", len(delta))
r0, v0 = delta[0].get_pv_np(0.0, frame="native")
r1, v1 = delta[1].get_pv_np(0.0, frame="native")
print("sat0 |r| [km], |v| [m/s]:", np.linalg.norm(r0)/1e3, np.linalg.norm(v0))
print("sat1 |r| [km], |v| [m/s]:", np.linalg.norm(r1)/1e3, np.linalg.norm(v1))


In [ ]:
# 3) Walker star: T/P/F = 24/6/2
star = build_walker_constellation(
    seed,
    total_satellites=24,
    num_planes=6,
    phasing=2,
    pattern="star",
    include_seed=False,
)

print("star count:", len(star))
r_last, v_last = star[-1].get_pv_np(0.0, frame="native")
print("last star sat |r| [km], |v| [m/s]:", np.linalg.norm(r_last)/1e3, np.linalg.norm(v_last))


In [ ]:
# 4) Numerical-construction Walker from a numerical seed
seed_num = Orbit.from_kepler_numerical(
    epoch=Time("2026-01-01T00:00:00", scale="utc"),
    a=7100e3,
    e=0.002,
    i=np.deg2rad(55.0),
    raan=np.deg2rad(30.0),
    argp=np.deg2rad(25.0),
    anomaly=np.deg2rad(5.0),
    anomaly_type="mean",
    gravity_degree=8,
    gravity_order=8,
    enable_drag=False,
    enable_third_body=False,
    enable_srp=False,
)

walker_num = build_walker_constellation(
    seed_num,
    total_satellites=12,
    num_planes=3,
    phasing=1,
    pattern="delta",
    include_seed=True,
    constructor="numerical",
    constructor_kwargs={
        "gravity_degree": 8,
        "gravity_order": 8,
        "enable_drag": False,
        "enable_third_body": False,
        "enable_srp": False,
    },
)

print("numerical walker count:", len(walker_num))
print("first propagator class:", walker_num[0].propagator.getClass().getSimpleName())


Next notebook: **04 - Interval Coverage**.
